In [56]:
import os
import subprocess
import time
import threading
from pymycobot.mycobot280 import MyCobot280
from pymycobot.genre import Angle, Coord

# ROS2 Jazzy 환경 활성화
env = subprocess.check_output(
    "bash -c 'source /opt/ros/jazzy/setup.bash && env'",
    shell=True,
    text=True
)

for line in env.splitlines():
    if '=' in line:
        key, value = line.split('=', 1)
        os.environ[key] = value

# ROS 도메인 아이디 설정
os.environ["ROS_DOMAIN_ID"] = "25"

print("ROS2 setup activated")
print("ROS_DOMAIN_ID =", os.environ["ROS_DOMAIN_ID"])

# 로봇 연결
mc = MyCobot280("/dev/ttyJETCOBOT", 1000000)
mc.thread_lock = True

print("로봇이 연결되었습니다.")

ROS2 setup activated
ROS_DOMAIN_ID = 25
로봇이 연결되었습니다.


In [37]:
# 로봇을 초기 위치로 리셋
initial_angles = [0, 0, 0, 0, 0, 0]
speed = 20

print("로봇을 초기 위치로 리셋합니다.")
mc.send_angles(initial_angles, speed)
mc.set_gripper_value(100, speed) # 그리퍼 열기
time.sleep(3) # 움직임이 완료될 때까지 대기
print("리셋 완료")

로봇을 초기 위치로 리셋합니다.
리셋 완료


In [17]:
# 모터 비활성화 (실행 전에 손으로 로봇을 잡고 시작)
print("전체 모터를 비활성화합니다.")
mc.release_all_servos()
time.sleep(8)  # 1 → 5초

# 모터 활성화
print("전체 모터를 활성화합니다.")
mc.focus_all_servos()
time.sleep(1)

전체 모터를 비활성화합니다.
전체 모터를 활성화합니다.


In [6]:
print("베이스 기준 xyz 좌표 : ", mc.get_coords())
print("관절의 각도 : ", mc.get_angles())

베이스 기준 xyz 좌표 :  [-8.1, 147.3, 230.5, -177.73, 2.78, 140.82]
관절의 각도 :  [118.74, 11.68, -74.35, -30.76, 1.05, -112.06]


In [55]:
# 지정된 좌표로 로봇팔 이동
import time
from pymycobot.mycobot280 import MyCobot280

mc = MyCobot280("/dev/ttyJETCOBOT", 1000000)
mc.thread_lock = True

pick_coords1 = [20.39, -7.29, -28.82, -50.44, 5.36, -110.65]
# pick_coords2 = [115.75, 0.7, -119.35, 18.54, 5.62, -113.11]


# [x(-280~280), y(-280~280), z(-70~523), rx(-180~180), ry(-180~180), rz(-180~180)] 단위: mm/도
# 좌표값
mc.send_coords(pick_coords1, 20, mode=1, _async=True)



while mc.is_moving():   # 이동 중이면 계속 대기
    time.sleep(0.1)

print("이동 완료!")
print("베이스 기준 xyz 좌표 : ", mc.get_coords())
print("관절의 각도         : ", mc.get_angles())

이동 완료!
베이스 기준 xyz 좌표 :  [56.5, -63.3, 408.4, -92.63, -1.02, -88.54]
관절의 각도         :  [0.7, -0.79, -0.61, -1.23, 0.7, -1.05]


In [ ]:
# 지정된 좌표로 로봇팔 이동
import time
from pymycobot.mycobot280 import MyCobot280

mc = MyCobot280("/dev/ttyJETCOBOT", 1000000)
mc.thread_lock = True



pick_angles = [20.39, -7.29, -28.82, -50.44, 5.36, -110.65] #INSPECTION_POSE
# pick_angles = [118.74, 11.68, -47.84, -30.76, 1.05, -112.06] #PICKUP_SLOT_approach
# pick_angles = [118.74, 11.68, -74.35, -30.76, 1.05, -112.06] #PICKUP_SLOT_place


# 관절값
mc.send_angles(pick_angles, 20)

while mc.is_moving():   # 이동 중이면 계속 대기
    time.sleep(0.1)

print("이동 완료!")
print("베이스 기준 xyz 좌표 : ", mc.get_coords())
print("관절의 각도         : ", mc.get_angles())

이동 완료!
베이스 기준 xyz 좌표 :  [161.3, -3.5, 266.9, -175.62, -1.58, 40.72]
관절의 각도         :  [20.83, -8.61, -29.97, -51.41, 4.65, -109.95]


In [28]:
mc.set_gripper_value(100, speed)

-1

In [ ]:
import time

print("이동 시작")
mc.send_coords([-51.8, -46.8, 410.9, 83.87, -47.83, 21.17], 20, mode=0)
print("send_coords 리턴됨")  # 이게 이동 완료 전에 찍히면 non-blocking

# is_moving으로 완료 대기
while True:
    moving = mc.is_moving()
    print(f"is_moving={moving}")
    if moving == 0:
        break
    time.sleep(0.1)

print("이동 완료")
print("현재 좌표:", mc.get_coords())

1. 그리퍼 열기
2. pick 이동
   현재 좌표: [3.5, -68.6, 412.4, -86.0, -43.61, -113.49]
3. 그리퍼 닫기
4. place 이동
   현재 좌표: [-55.0, -62.5, 375.0, 93.68, -47.88, 13.94]
5. 그리퍼 열기


In [1]:
from pymycobot.mycobot280 import MyCobot280
import time

mc = MyCobot280('/dev/ttyJETCOBOT', 1000000)
mc.thread_lock = True

mc.power_on()
time.sleep(1)

print("Before angles:", mc.get_angles())
print("Before encoders:", mc.get_encoders())

# 매우 주의:
# 1. 로봇을 손으로 받친다.
# 2. 서보 토크를 풀고,
# 3. 각 joint의 물리적인 zero position 눈금선을 정확히 맞춘다.
# 4. 그 상태에서 해당 joint를 calibration 한다.

mc.release_all_servos()
time.sleep(2)

input("모든 joint zero mark를 물리적으로 맞춘 뒤 Enter를 누르세요...")

for servo_id in range(1, 7):
    ret = mc.set_servo_calibration(servo_id)
    print(f"servo {servo_id} calibration result:", ret)
    time.sleep(0.5)

mc.power_on()
time.sleep(1)

print("After angles:", mc.get_angles())
print("After encoders:", mc.get_encoders())

mc.send_angles([0, 0, 0, 0, 0, 0], 20)
time.sleep(5)
print("Final angles:", mc.get_angles())

Before angles: [0.26, -0.52, -0.61, -0.96, -0.52, -0.7]
Before encoders: [2045, 2054, 2041, 2059, 2054, 2056]
servo 1 calibration result: -1
servo 2 calibration result: -1
servo 3 calibration result: -1
servo 4 calibration result: -1
servo 5 calibration result: -1
servo 6 calibration result: -1
After angles: [0.17, 0.0, -0.08, -0.17, 0.61, 0.08]
After encoders: [2046, 2048, 2047, 2050, 2041, 2047]
Final angles: [0.52, -0.17, -0.43, -0.52, 0.7, 0.08]
